## 1. Chargement : parties de 3 minutes

In [ ]:
import pandas as pd
import polars as pl
from utils.pgn import parse_moves

In [ ]:
df = pd.read_parquet("../../Data/sample_10k.parquet")
df.shape

In [ ]:
df = df[df["TimeControl"].isin(["180+0", "180+2"])]
df.shape

In [ ]:
df["Movetext"].iloc[0][:160]

## 2. Nouveau format des coups : {san, clk, eval}

In [ ]:
df = df.drop(columns=["moves", "n_moves"])
df.shape

In [ ]:
df = parse_moves(pl.from_pandas(df)).to_pandas()
df.shape

In [ ]:
df["moves"][0][:5]

## 3. Parties avec une eval

In [ ]:
coups = df["moves"].explode()
coups.head()

In [ ]:
evals = coups.str["eval"]
evals.notna().sum()

In [ ]:
a_eval = evals.notna().groupby(level=0).any()
a_eval.sum()

## 4. Parties à retirer : bots, fins anormales, absents

In [ ]:
bot = (df["WhiteTitle"] == "BOT") | (df["BlackTitle"] == "BOT")
bot.sum()

In [ ]:
df["Termination"].value_counts()

In [ ]:
mauvaise_fin = df["Termination"].isin(["Abandoned", "Rules infraction"])
mauvaise_fin.sum()

In [ ]:
au_temps = df["Termination"] == "Time forfeit"
au_temps.sum()

In [ ]:
# temps restant au perdant à son dernier coup (avant-dernier coup de la partie)
temps_restant = pd.to_timedelta(df["moves"].str[-2].str["clk"]).dt.total_seconds()
temps_restant[au_temps].describe()

In [ ]:
# absent = perdu au temps alors qu'il lui restait plus de 30 s
absent = au_temps & (temps_restant > 30)
absent.sum()

In [ ]:
pipeline_retirer = bot | mauvaise_fin | absent
pipeline_retirer.sum()

In [ ]:
df = df.drop(df[pipeline_retirer].index)
df.shape

## 5. Une ligne par joueur

In [ ]:
# Étape 13 : identifiant de partie
df["game_id"] = df["Site"].str[-8:]

# Étape 14 : table des Blancs (coups 0, 2, 4...)
blancs = pd.DataFrame({
    "game_id": df["game_id"],
    "joueur": df["White"],
    "couleur": "blanc",
    "elo": df["WhiteElo"],
    "rating_diff": df["WhiteRatingDiff"],
    "cadence": df["TimeControl"],
    "resultat": df["Result"],
    "termination": df["Termination"],
    "eco": df["ECO"],
    "coups": df["moves"].str[0::2],
})

# Étape 15 : table des Noirs (coups 1, 3, 5...)
noirs = pd.DataFrame({
    "game_id": df["game_id"],
    "joueur": df["Black"],
    "couleur": "noir",
    "elo": df["BlackElo"],
    "rating_diff": df["BlackRatingDiff"],
    "cadence": df["TimeControl"],
    "resultat": df["Result"],
    "termination": df["Termination"],
    "eco": df["ECO"],
    "coups": df["moves"].str[1::2],
})

In [ ]:
joueurs = pd.concat([blancs, noirs], ignore_index=True)
joueurs.shape

In [ ]:
joueurs.head()

## 6. Elo provisoires (|rating_diff| > 15)

In [ ]:
ecart_elo = joueurs["rating_diff"].abs()
ecart_elo.describe(percentiles=[0.5, 0.9, 0.95, 0.99])

In [ ]:
import matplotlib.pyplot as plt

ecart_elo.clip(upper=40).plot.hist(bins=41, range=(-0.5, 40.5), log=True)
plt.axvline(15.5, color="red", ls="--", label="seuil 15")
plt.xlabel("|rating_diff| (40 = 40 et plus)")
plt.ylabel("nombre de joueurs")
plt.legend()
plt.show()

In [ ]:
joueurs.groupby(ecart_elo.clip(upper=40))["elo"].agg(["count", "mean"]).round()

In [ ]:
joueurs.groupby(ecart_elo > 15)["elo"].agg(["count", "mean"]).round()

In [ ]:
# 1. les joueurs provisoires (écart de plus de 15 points)
provisoire = joueurs["rating_diff"].abs() > 15
joueurs = joueurs.drop(joueurs[provisoire].index)

# 2. les joueurs sans rating_diff (case vide)
joueurs = joueurs.dropna(subset=["rating_diff"])

joueurs.shape

## 7. Pendule : berserk et temps ajouté

In [ ]:
coups = joueurs["coups"].explode()
clk = pd.to_timedelta(coups.str["clk"]).dt.total_seconds()
clk.head()

In [ ]:
debut = clk.groupby(level=0).first()
debut.value_counts().head()

In [ ]:
berserk = debut < 170
berserk.sum()

In [ ]:
inc = joueurs["cadence"].str.split("+").str[1].astype(int)
inc.value_counts()

In [ ]:
ecart = clk.groupby(level=0).diff()
ecart.head(6)

In [ ]:
temps = -ecart + inc
temps.head(6)

In [ ]:
temps.min()

In [ ]:
temps_ajoute = (temps < 0).groupby(level=0).any()
temps_ajoute.sum()

In [ ]:
joueurs_retirer = berserk | temps_ajoute
joueurs = joueurs.drop(joueurs[joueurs_retirer].index)
joueurs.shape

## 8. Recalcul sur la table propre

In [ ]:
coups = joueurs["coups"].explode()
clk = pd.to_timedelta(coups.str["clk"]).dt.total_seconds()
inc = joueurs["cadence"].str.split("+").str[1].astype(int)
ecart = clk.groupby(level=0).diff()
temps = -ecart + inc
temps.min()

In [ ]:
num = coups.groupby(level=0).cumcount()
num.head(4)